In [1]:
import pandas as pd
import numpy as np
import glob
import os

from itertools import combinations
from scipy.stats import beta

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score
import xgboost as xgb

In [8]:
# data path
DATA_PATH = r"D:\lol draft analyzer - datascientest\AA- toutes les donnees au propre - lecture-ecriture"

# Games
GAMES_PATH = f"{DATA_PATH}\\les matchs\\200k_games\\draft_simple.csv"
df_games = pd.read_csv(GAMES_PATH)

# # Champions
# CHAMPS_PATH = f"{DATA_PATH}\\les stats champions\\champions_15.1.1_15.24.1.csv"
# df_champs = pd.read_csv(CHAMPS_PATH, encoding="utf-8")

# # WR simples
# SIMPLE_WR_PATH = f"{DATA_PATH}\\les winrates simples\\données raffinées\\df_Simple_WR_FULL.csv"
# df_simple_wr = pd.read_csv(SIMPLE_WR_PATH, encoding="utf-8")

# # WR complets
# # COMPLET_WR_PATH = f"{DATA_PATH}\\les winrates matchups\\WR_complet.csv"
# # df_complet_wr = pd.read_csv(COMPLET_WR_PATH, encoding="utf-8")


C:\Users\samue\AppData\Local\Temp\ipykernel_30496\3991117897.py:6: DtypeWarning: Columns (10,11,12,13,14,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_games = pd.read_csv(GAMES_PATH)


In [9]:
df_games.head()

,match_id,serveur,patch,elo,blue_side_win,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion,...,blue_ban_1,blue_ban_2,blue_ban_3,blue_ban_4,blue_ban_5,red_ban_1,red_ban_2,red_ban_3,red_ban_4,red_ban_5
0,EUW1_7517331727,euw1,15.17.708.5788,HIGH_ELO,False,Sett,Talon,Diana,Jinx,Rell,...,Lulu,Fiora,Rengar,Vex,Poppy,Dr. Mundo,Garen,Milio,Zoe,Draven
1,EUW1_7509322616,euw1,15.17.706.7412,HIGH_ELO,False,Mordekaiser,Qiyana,Yasuo,Corki,Nami,...,Yunara,Pyke,Gwen,Kayle,Smolder,Master Yi,Pantheon,Twitch,Draven,Fiora
2,EUW1_7509243675,euw1,15.17.706.7412,HIGH_ELO,False,Smolder,Kindred,Galio,Jinx,Milio,...,Dr. Mundo,Rengar,Fiora,Master Yi,Yasuo,Master Yi,Twitch,Akali,Draven,Aatrox
3,EUW1_7509193063,euw1,15.17.706.7412,HIGH_ELO,False,Warwick,Nidalee,Akali,Yunara,Rakan,...,Sivir,Qiyana,Naafiri,Draven,Twitch,Fiddlesticks,Blitzcrank,Mel,Volibear,Darius
4,EUW1_7508909082,euw1,15.17.706.7412,HIGH_ELO,False,Aurora,XinZhao,Twitch,Yunara,Rakan,...,Draven,Kassadin,Shaco,Ahri,Senna,Nocturne,Evelynn,Galio,Yasuo,Riven


In [2]:
def load_all_csvs(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "**/*.csv"), recursive=True)
    df_list = [pd.read_csv(f) for f in all_files]
    return pd.concat(df_list, ignore_index=True)


PATH = r"D:\lol draft analyzer - datascientest\AA- toutes les donnees au propre - lecture-ecriture\les winrates matchups\TOUTTOUT15.24"
matchups_df = load_all_csvs(os.path.join(PATH, "matchups/"))
synergies_df = load_all_csvs(os.path.join(PATH, "synergies/"))

In [3]:
matchups_df.head()
synergies_df.head()

,champion,role,role_play_ratio,tier,rank,winrate,pickrate,banrate,nb_games_analyzed,url,...,synergy_top_64_games,synergy_top_64_lane_quality,synergy_top_65_name,synergy_top_65_winrate,synergy_top_65_games,synergy_top_65_lane_quality,synergy_top_66_name,synergy_top_66_winrate,synergy_top_66_games,synergy_top_66_lane_quality
0,Nami,sup,99.8%,S+,2 / 39,52.3 %,10.4 %,2.2 %,2491832,https://dpm.lol/champions/Nami/build?lane=util...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dr.Mundo,top,60.3%,S,2 / 50,52.0 %,6.2 %,13.5 %,1495213,https://dpm.lol/champions/DrMundo/build?lane=t...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Viego,jun,97.5%,S,2 / 52,50.9 %,11.0 %,9.7 %,2645201,https://dpm.lol/champions/Viego/build?lane=jun...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Jax,jun,51.6%,S,3 / 52,52.0 %,5.9 %,11.6 %,1421552,https://dpm.lol/champions/Jax/build?lane=jungl...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dr.Mundo,jun,35.0%,S,4 / 52,52.6 %,3.6 %,13.5 %,866732,https://dpm.lol/champions/DrMundo/build?lane=j...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def clean_percent(col):
    return col.str.replace("%", "", regex=False).astype(float) / 100

# IL FAUT AJOUTER LES WINRATES SIMPLES POUR COMBLER LES DONNEES MANQUANTES

matchups_df["winrate"] = clean_percent(matchups_df["winrate"])
matchups_df["role_play_ratio"] = clean_percent(matchups_df["role_play_ratio"])
matchups_df["pickrate"] = clean_percent(matchups_df["pickrate"])
matchups_df["banrate"] = clean_percent(matchups_df["banrate"])

synergies_df["winrate"] = clean_percent(synergies_df["winrate"])
synergies_df["role_play_ratio"] = clean_percent(synergies_df["role_play_ratio"])
synergies_df["pickrate"] = clean_percent(synergies_df["pickrate"])  
synergies_df["banrate"] = clean_percent(synergies_df["banrate"])


ValueError: could not convert string to float: '-'

In [5]:
def explode_matchups(df, lane_prefix):
    rows = []
    
    for _, row in df.iterrows():
        champ = row["champion"]
        role = row["role"]
        
        for i in range(1, 5):
            name_col = f"{lane_prefix}_{i}_name"
            wr_col = f"{lane_prefix}_{i}_winrate"
            games_col = f"{lane_prefix}_{i}_games"
            
            if name_col in df.columns and pd.notna(row[name_col]):
                rows.append({
                    "champion": champ,
                    "role": role,
                    "opponent": row[name_col],
                    "winrate": float(str(row[wr_col]).replace("%","")) / 100,
                    "games": row[games_col]
                })
                
    return pd.DataFrame(rows)

In [6]:
adc_matchups = explode_matchups(matchups_df, "matchup_adc")
# top_matchups = explode_matchups(matchups_df, "matchup_top")
# mid_matchups = explode_matchups(matchups_df, "matchup_mid")
# jungle_matchups = explode_matchups(matchups_df, "matchup_jungle")
# support_matchups = explode_matchups(matchups_df, "matchup_support")

In [7]:
adc_matchups.head()

,champion,role,opponent,winrate,games
0,Nami,sup,Corki,0.5634,16110.0
1,Nami,sup,Kalista,0.5553,16231.0
2,Nami,sup,Varus,0.5537,44692.0
3,Nami,sup,Yunara,0.5503,80587.0
4,Dr.Mundo,top,Corki,0.5557,7976.0


In [10]:
def get_matchup_wr(champ, opponent, matchup_table):
    row = matchup_table[
        (matchup_table["champion"] == champ) &
        (matchup_table["opponent"] == opponent)
    ]
    
    if len(row) == 0:
        return 0.5
    
    return row.iloc[0]["winrate"]

In [11]:
lanes = ["top", "jungle", "mid", "adc", "support"]

def build_features(games, matchup_table):
    feature_rows = []
    
    for _, row in games.iterrows():
        features = {}
        
        for lane in lanes:
            blue = row[f"blue_{lane}_champion"]
            red = row[f"red_{lane}_champion"]
            
            wr = get_matchup_wr(blue, red, matchup_table)
            features[f"{lane}_matchup_wr"] = wr
        
        feature_rows.append(features)
    
    return pd.DataFrame(feature_rows)

In [ ]:
X = build_features(df_games, adc_matchups)  # à généraliser
y = df_games["blue_side_win"].astype(int)

X = X.fillna(0.5)

In [ ]:
#XGBOOST

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
)

model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:,1]

print("LogLoss:", log_loss(y_test, preds))
print("AUC:", roc_auc_score(y_test, preds))

In [ ]:
# fonction beta(a, b):

In [ ]:
def beta_from_stats(p, n):
    # p = WR
    # n = nb games 
    alpha = (p * n) / 100
    beta_param = ((1 - p) * n) / 100
    return alpha, beta_param